In [1]:
# --- Configuración de entorno ---

# Añade el directorio raíz al path para que Python encuentre tus módulos
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../..")))  # Sube dos niveles hasta /src

# Importa configuraciones y librerías globales
from config import *
from utils import *

import importlib
import utils

importlib.reload(utils)

<module 'utils' from 'D:\\MASTER EN CIENCIA DE DATOS\\TFM\\TrabajoFinal\\ivst-tfm\\src\\utils.py'>

# Carga de los ficheros catastrales
Se descarga el conjunto de datos catastrales de todas las provincias de Castilla y León desde la web oficial del Catastro Público:
https://www.sedecatastro.gob.es/DescargaDatos/SECDescargaCartoSF.aspx


In [2]:
df_raw_all = load_all_cat_provinces(os.path.join(DATA_INPUTS_DH, "Ficheros CAT"))

print(df_raw_all.head())
print(df_raw_all.shape)
print(df_raw_all["record_type"].value_counts())

df_raw_all["record_type"] = df_raw_all["record_type"].astype("category")

Detectadas 9 provincias. Iniciando proceso de integracion...


Procesando provincias: 100%|█████████████████████████████████████████████████████████████| 9/9 [01:12<00:00,  8.07s/it]


Consolidacion completada: 13879803 registros cargados.
                                          raw_record record_type  \
0  01G05       Ávila                      2025081...          01   
1  110000             000 05001  4941114UL6344S  ...          11   
2  130000             000 05001UR4941114UL6344S00...          13   
3  130000             000 05001UR4941114UL6344S00...          13   
4  140000             000 05001  4941114UL6344S00...          14   

              source_file provincia  
0  05001U_14082025.CAT.gz     Avila  
1  05001U_14082025.CAT.gz     Avila  
2  05001U_14082025.CAT.gz     Avila  
3  05001U_14082025.CAT.gz     Avila  
4  05001U_14082025.CAT.gz     Avila  
(13879803, 4)
record_type
14    5872575
15    3542214
13    2241865
11    1835606
16     193633
17     189414
01       2248
90       2248
Name: count, dtype: int64


In [3]:
# Obtengo fincas para tener la localización y año de construcción
fincas = df_raw_all.loc[df_raw_all.record_type == "11"]
parsed_rows = []

print(f"Procesando {len(fincas)} registros tipo 11...")

for raw in tqdm(fincas["raw_record"], desc="Parseando fincas (registro 11)", unit="rec"):
    parsed_rows.append(parse_cat_record_11(raw))

fincas_parseadas = pd.DataFrame(parsed_rows)

print(fincas_parseadas.info())
fincas_parseadas.sample(5)

Procesando 1835606 registros tipo 11...


Parseando fincas (registro 11): 100%|███████████████████████████████████| 1835606/1835606 [00:08<00:00, 217419.99rec/s]


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1835606 entries, 0 to 1835605
Data columns (total 10 columns):
 #   Column                       Dtype 
---  ------                       ----- 
 0   id_parcela                   object
 1   superficie_finca_m2          object
 2   superficie_construida_total  object
 3   superficie_sobre_rasante     object
 4   superficie_bajo_sasante      object
 5   superficie_cubierta          object
 6   X                            object
 7   Y                            object
 8   CP                           object
 9   sist_coord                   object
dtypes: object(10)
memory usage: 140.0+ MB
None


,id_parcela,superficie_finca_m2,superficie_construida_total,superficie_sobre_rasante,superficie_bajo_sasante,superficie_cubierta,X,Y,CP,sist_coord
924580,1001707UM7610S,0000001128,0000000,0000000,0000000,0000000,037083942,0465994009,00000,EPSG:25830
405714,002400100VN81E,0000000300,0000300,0000300,0000000,0000300,048331480,0471498002,00000,EPSG:25830
734988,8734111TM5883S,0000000463,0000000,0000000,0000000,0000000,025859030,0468316820,24767,EPSG:25830
1534653,3430430UM5233S,0000000161,0000231,0000210,0000021,0000126,035329272,0462275705,00000,EPSG:25830
826321,2710712TN7121S,0000000137,0000260,0000260,0000000,0000000,027262590,0471086237,24392,EPSG:25830


In [4]:
# Obtengo bienes para obtener superficie y tipo de bien
bienes = df_raw_all.loc[df_raw_all.record_type == "15"]
parsed_rows = []

print(f"Procesando {len(bienes)} registros tipo 15...")

for raw in tqdm(bienes["raw_record"], desc="Parseando bienes (registro 15)", unit="rec"):
    parsed_rows.append(parse_cat_record_15(raw))

bienes_parseados = pd.DataFrame(parsed_rows)

# Unicamente me quedo con viviendas
bienes_parseados = bienes_parseados[bienes_parseados["uso"] == "V"]
print(bienes_parseados.info())
bienes_parseados.sample(5)

Procesando 3542214 registros tipo 15...


Parseando bienes (registro 15): 100%|███████████████████████████████████| 3542214/3542214 [00:31<00:00, 114001.41rec/s]


<class 'pandas.core.frame.DataFrame'>
Index: 1766996 entries, 2 to 3542207
Data columns (total 6 columns):
 #   Column                                 Dtype 
---  ------                                 ----- 
 0   uso                                    object
 1   id_parcela                             object
 2   id_bien                                object
 3   superficie_inmueble_division_vertical  object
 4   superficie_inmueble_en_solar           object
 5   Antiguedad                             object
dtypes: object(6)
memory usage: 94.4+ MB
None


,uso,id_parcela,id_bien,superficie_inmueble_division_vertical,superficie_inmueble_en_solar,Antiguedad
3454932,V,2826206QG4022N,0001,0000000159,0000001543,2009
1197655,V,7836402PH9173N,0008,0000000094,0000000180,1972
1268099,V,4383805TN8148S,0004,0000000088,0000000379,1970
1485960,V,4365202TM7846N,0001,0000000150,0000000213,1950
1456538,V,0340305TN9204S,0012,0000000098,0000000421,1995


In [5]:
# Obtengo bienes para obtener superficie y tipo de bien
reformas = df_raw_all.loc[df_raw_all.record_type == "14"]
parsed_rows = []

print(f"Procesando {len(reformas)} registros tipo 14...")
for raw in tqdm(reformas["raw_record"], desc="Parseando reformas (registro 14)", unit="rec"):
    parsed_rows.append(parse_cat_record_14(raw))

reformas_parseados = pd.DataFrame(parsed_rows)

# Unicamente me quedo con los que tengan reformas
reformas_parseados = reformas_parseados[reformas_parseados["tipo_reforma"] != " "]

print(reformas_parseados.info())
reformas_parseados.sample(5)

Procesando 5872575 registros tipo 14...


Parseando reformas (registro 14): 100%|██████████████████████████████████| 5872575/5872575 [01:13<00:00, 79733.85rec/s]


<class 'pandas.core.frame.DataFrame'>
Index: 825026 entries, 36 to 5872574
Data columns (total 4 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   tipo_reforma  825026 non-null  object
 1   id_parcela    825026 non-null  object
 2   id_bien       825026 non-null  object
 3   año_reforma   825026 non-null  object
dtypes: object(4)
memory usage: 31.5+ MB
None


,tipo_reforma,id_parcela,id_bien,año_reforma
3924933,E,2037901VL3523N,0001,1987
1744131,O,8716412QH1381N,0001,1992
4506808,E,0747128UL1804N,0001,1982
4138640,E,7815116WM6071N,0001,2010
3300134,E,9418305TL8591N,0001,1999


In [6]:
# Normalización de coordenadas X e Y procedentes del Catastro

# Eliminamos espacios por si vienen alineados a la derecha
fincas_parseadas["X"] = fincas_parseadas["X"].str.strip()
fincas_parseadas["Y"] = fincas_parseadas["Y"].str.strip()

# Convertimos a entero (maneja nulos de forma segura)
fincas_parseadas["X"] = pd.to_numeric(fincas_parseadas["X"], errors="coerce").astype("Int64")
fincas_parseadas["Y"] = pd.to_numeric(fincas_parseadas["Y"], errors="coerce").astype("Int64")

# Escalamos a coordenadas reales (división entre 100)
fincas_parseadas["X"] = fincas_parseadas["X"] / 100
fincas_parseadas["Y"] = fincas_parseadas["Y"] / 100


In [7]:
# Elimino lo que tienen coordenadas 0000 (asumo errores de datos)
fincas_parseadas = fincas_parseadas[fincas_parseadas["sist_coord"] != "0000000000"]

fincas_parseadas = asignar_cusec_catastro(
    fincas_parseadas
)

print(fincas_parseadas.info())
fincas_parseadas.sample(5)


Procesando CRS: EPSG:25829 (365317 filas)

Procesando CRS: EPSG:25830 (1468791 filas)
<class 'pandas.core.frame.DataFrame'>
Index: 1834108 entries, 0 to 1835605
Data columns (total 11 columns):
 #   Column                       Dtype  
---  ------                       -----  
 0   id_parcela                   object 
 1   superficie_finca_m2          object 
 2   superficie_construida_total  object 
 3   superficie_sobre_rasante     object 
 4   superficie_bajo_sasante      object 
 5   superficie_cubierta          object 
 6   X                            Float64
 7   Y                            Float64
 8   CP                           object 
 9   sist_coord                   object 
 10  CUSEC                        object 
dtypes: Float64(2), object(9)
memory usage: 235.9+ MB
None


,id_parcela,superficie_finca_m2,superficie_construida_total,superficie_sobre_rasante,superficie_bajo_sasante,superficie_cubierta,X,Y,CP,sist_coord,CUSEC
450337,3382502VM1838S,0000000116,0000156,0000156,0000000,0000116,413270.41,4668071.29,00000,EPSG:25830,0946701001
1322236,3705108VL2930N,0000000177,0000270,0000270,0000000,0000108,420448.17,4583489.81,00000,EPSG:25830,4018301001
809636,2030609PH9323S,0000000143,0000000,0000000,0000000,0000000,691957.93,4732831.28,24430,EPSG:25829,2419601001
647836,0273804QH1407S,0000000060,0000117,0000117,0000000,0000060,710118.8,4747065.1,24495,EPSG:25829,2410901001
1717481,3029520QG3632N,0000000151,0000052,0000032,0000020,0000032,732917.03,4662732.88,00000,EPSG:25829,4912001001


In [8]:
fincas_parseadas.sample(10)

,id_parcela,superficie_finca_m2,superficie_construida_total,superficie_sobre_rasante,superficie_bajo_sasante,superficie_cubierta,X,Y,CP,sist_coord,CUSEC
815763,9251620UN0495S,0000000024,0000024,0000024,0000000,0000024,309210.73,4744883.46,24869,EPSG:25830,2419901001
1184905,1971403PF9017S,0000000060,0000000,0000000,0000000,0000000,691801.02,4506832.14,00000,EPSG:25829,3735901001
470504,6024902TM7762S,0000000202,0000239,0000239,0000000,0000140,275938.45,4672213.26,24796,EPSG:25830,2400501001
1348111,1152339UL8115S,0000000083,0000128,0000080,0000048,0000048,381052.19,4515049.3,40150,EPSG:25830,4022501001
1457846,42287A00505421,0000000028,0000000,0000000,0000000,0000000,529894.41,4604767.06,42294,EPSG:25830,4218101001
1312802,9300198VL5790S,0000003301,0000000,0000000,0000000,0000000,459226.78,4569886.83,40500,EPSG:25830,4017001001
293621,8979519VM3487N,0000000026,0000026,0000026,0000000,0000026,438778.76,4647654.57,09349,EPSG:25830,0919401002
512100,24027A21115177,0000000061,0000000,0000000,0000000,0000000,270247.36,4702343.27,24248,EPSG:25830,2402601001
1264096,1484401VL1318S,0000000114,0000114,0000114,0000000,0000114,411196.11,4538228.63,00000,EPSG:25830,4007701001
1704689,2452413TL8925S,0000000158,0000128,0000128,0000000,0000128,282399.18,4595003.45,00000,EPSG:25830,4910301001


In [9]:
fincas_parseadas["CUSEC"].isna().sum()

np.int64(94)

In [10]:
# Intento unir los que faltan por CP:
fincas_parseadas = asignar_cusec_por_cp(fincas_parseadas)
fincas_parseadas["CUSEC"].isna().sum()

✔️ 1834108 registros procesados
   🔹 CUSEC completados: 94 | Sin asignar: 0
   🔹 Seccion_id completados: 873259 | Sin asignar: 960849


np.int64(0)

In [11]:
(fincas_parseadas["CUSEC"] == "0000000nan").sum()

np.int64(77)

In [12]:
# Borro estas que me han quedado ya que es tamaño residual
fincas_parseadas = fincas_parseadas[fincas_parseadas["CUSEC"] != "0000000nan"]

In [13]:
cols_drop = ["X", "Y", "sist_coord", "CP"]
cols_drop = [c for c in cols_drop if c in fincas_parseadas.columns]

fincas_parseadas = fincas_parseadas.drop(columns=cols_drop)
print(fincas_parseadas.info())
fincas_parseadas.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 1834031 entries, 0 to 1835605
Data columns (total 8 columns):
 #   Column                       Dtype 
---  ------                       ----- 
 0   id_parcela                   object
 1   superficie_finca_m2          object
 2   superficie_construida_total  object
 3   superficie_sobre_rasante     object
 4   superficie_bajo_sasante      object
 5   superficie_cubierta          object
 6   CUSEC                        object
 7   Seccion_id                   Int64 
dtypes: Int64(1), object(7)
memory usage: 127.7+ MB
None


,id_parcela,superficie_finca_m2,superficie_construida_total,superficie_sobre_rasante,superficie_bajo_sasante,superficie_cubierta,CUSEC,Seccion_id
882043,7872942TN8177S,0000000431,0000000,0000000,0000000,0000000,2408909003,<NA>
1347567,1900603VL4510S,0000000220,0000000,0000000,0000000,0000000,4022401001,<NA>
666418,6755107PH9165N,0000000746,0000021,0000021,0000000,0000021,2411505004,1116
591276,8428939PG9882N,0000000118,0000118,0000118,0000000,0000118,2406701001,969
1485788,8104111UM5080S,0000000514,0000000,0000000,0000000,0000000,4702301001,<NA>


In [14]:
# Union de datos

cols_numericas_bienes = [
    "superficie_inmueble_division_vertical",
    "superficie_inmueble_en_solar",
]

for c in cols_numericas_bienes:
    bienes_parseados[c] = (
        bienes_parseados[c]
        .astype(str)
        .str.lstrip("0")
        .replace("", "0")
        .astype(int)
    )

print(bienes_parseados.info())
bienes_parseados.sample(5)

<class 'pandas.core.frame.DataFrame'>
Index: 1766996 entries, 2 to 3542207
Data columns (total 6 columns):
 #   Column                                 Dtype 
---  ------                                 ----- 
 0   uso                                    object
 1   id_parcela                             object
 2   id_bien                                object
 3   superficie_inmueble_division_vertical  int64 
 4   superficie_inmueble_en_solar           int64 
 5   Antiguedad                             object
dtypes: int64(2), object(4)
memory usage: 94.4+ MB
None


,uso,id_parcela,id_bien,superficie_inmueble_division_vertical,superficie_inmueble_en_solar,Antiguedad
857193,V,3797501VM4839N,0047,77,2156,1988
1379917,V,2259212TM9825N,0088,93,1499,1996
1641592,V,6220829UM7462S,0001,176,118,1994
1512773,V,9586203TN8198N,0001,57,315,1960
3506438,V,1795001TL7919N,0130,137,1137,2001


In [15]:
# Integro el CUSEC en los bienes inmuebles:
bienes_parseados = bienes_parseados.merge(
    fincas_parseadas,
    on="id_parcela",
    how="left"
)

In [16]:
print(bienes_parseados.info())
bienes_parseados.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1767127 entries, 0 to 1767126
Data columns (total 13 columns):
 #   Column                                 Dtype 
---  ------                                 ----- 
 0   uso                                    object
 1   id_parcela                             object
 2   id_bien                                object
 3   superficie_inmueble_division_vertical  int64 
 4   superficie_inmueble_en_solar           int64 
 5   Antiguedad                             object
 6   superficie_finca_m2                    object
 7   superficie_construida_total            object
 8   superficie_sobre_rasante               object
 9   superficie_bajo_sasante                object
 10  superficie_cubierta                    object
 11  CUSEC                                  object
 12  Seccion_id                             Int64 
dtypes: Int64(1), int64(2), object(10)
memory usage: 177.0+ MB
None


,uso,id_parcela,id_bien,superficie_inmueble_division_vertical,superficie_inmueble_en_solar,Antiguedad,superficie_finca_m2,superficie_construida_total,superficie_sobre_rasante,superficie_bajo_sasante,superficie_cubierta,CUSEC,Seccion_id
1120359,V,6167101TL7366E,0046,91,836,1973,0000000836,0005403,0005403,0000000,0000650,3727403008,1897
517855,V,9046604TM8794N,0001,318,323,1900,0000000323,0000318,0000318,0000000,0000196,2408801001,<NA>
1752933,V,1289701TL7918N,0048,105,462,2001,0000000462,0002811,0001887,0000924,0000462,4927501003,3424
1718386,V,8842401QG3084S,0001,150,75,1991,0000000075,0000150,0000150,0000000,0000075,4924001001,3413
703673,V,8108402TN8280N,0060,160,1870,1998,0000001870,0012916,0009878,0003038,0000315,2408903009,1019


In [17]:
bienes_parseados["CUSEC"].isna().sum()

np.int64(375)

In [18]:
# Elimino las que han quedado sin CUSEC
bienes_parseados = bienes_parseados[bienes_parseados["CUSEC"].notna()]

In [19]:
# Integro el año de última reforma
reformas_parseados["año_reforma"] = (
    reformas_parseados["año_reforma"]
    .str.extract(r"(\d{4})")
    .astype(float)
    .astype("Int64")
)

reformas_ultimas = (
    reformas_parseados
    .groupby(["id_parcela", "id_bien"], as_index=False)["año_reforma"]
    .max()
    .rename(columns={"año_reforma": "anio_ultima_reforma"})
)

bienes_parseados = bienes_parseados.merge(
    reformas_ultimas,
    on=["id_parcela", "id_bien"],
    how="left"
)

print(bienes_parseados.info())
bienes_parseados.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1766752 entries, 0 to 1766751
Data columns (total 14 columns):
 #   Column                                 Dtype 
---  ------                                 ----- 
 0   uso                                    object
 1   id_parcela                             object
 2   id_bien                                object
 3   superficie_inmueble_division_vertical  int64 
 4   superficie_inmueble_en_solar           int64 
 5   Antiguedad                             object
 6   superficie_finca_m2                    object
 7   superficie_construida_total            object
 8   superficie_sobre_rasante               object
 9   superficie_bajo_sasante                object
 10  superficie_cubierta                    object
 11  CUSEC                                  object
 12  Seccion_id                             Int64 
 13  anio_ultima_reforma                    Int64 
dtypes: Int64(2), int64(2), object(10)
memory usage: 192.1+ MB
None


,uso,id_parcela,id_bien,superficie_inmueble_division_vertical,superficie_inmueble_en_solar,Antiguedad,superficie_finca_m2,superficie_construida_total,superficie_sobre_rasante,superficie_bajo_sasante,superficie_cubierta,CUSEC,Seccion_id,anio_ultima_reforma
839111,V,3877713UM6537N,0001,419,650,1965,0000000650,0000419,0000419,0000000,0000419,3423701001,1557,<NA>
769428,V,9796902TN8199N,0001,211,92,1961,0000000092,0000211,0000211,0000000,0000091,2408910001,1080,2000
1241196,V,7718014VL0371N,0017,186,8848,1986,0000008848,0011848,0008850,0002998,0001404,4019405006,2311,<NA>
1049863,V,6279116TL7367G,0001,412,198,2018,0000000198,0000412,0000412,0000000,0000189,3727403001,1869,2018
293286,V,4886310VM9948N,0001,440,220,1989,0000000220,0000440,0000440,0000000,0000220,0930701001,<NA>,<NA>


In [20]:
cols_drop = ["uso", "superficie_inmueble_en_solar"]
cols_drop = [c for c in cols_drop if c in bienes_parseados.columns]
bienes_parseados = bienes_parseados.drop(columns=cols_drop)

In [21]:
print(bienes_parseados.info())
bienes_parseados.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1766752 entries, 0 to 1766751
Data columns (total 12 columns):
 #   Column                                 Dtype 
---  ------                                 ----- 
 0   id_parcela                             object
 1   id_bien                                object
 2   superficie_inmueble_division_vertical  int64 
 3   Antiguedad                             object
 4   superficie_finca_m2                    object
 5   superficie_construida_total            object
 6   superficie_sobre_rasante               object
 7   superficie_bajo_sasante                object
 8   superficie_cubierta                    object
 9   CUSEC                                  object
 10  Seccion_id                             Int64 
 11  anio_ultima_reforma                    Int64 
dtypes: Int64(2), int64(1), object(9)
memory usage: 165.1+ MB
None


,id_parcela,id_bien,superficie_inmueble_division_vertical,Antiguedad,superficie_finca_m2,superficie_construida_total,superficie_sobre_rasante,superficie_bajo_sasante,superficie_cubierta,CUSEC,Seccion_id,anio_ultima_reforma
1221535,6429701VL0362N,0085,107,1952,0000002727,0010832,0010832,0000000,0002686,4019404005,2317,<NA>
1324037,3431301WM4233N,0050,107,2000,0000001221,0002334,0001749,0000585,0000589,4217302014,2544,<NA>
192616,9550423VM3395S,0001,300,1980,0000000175,0000300,0000300,0000000,0000113,0903301001,<NA>,<NA>
122975,2653024UK7725S,0001,77,1980,0000000207,0000529,0000529,0000000,0000136,0524101001,<NA>,<NA>
1711891,4939106TM7643N,0001,327,1922,0000000394,0000327,0000327,0000000,0000243,4922001001,3370,2012


In [22]:
# Aseguramos tipos numéricos
bienes_parseados["Antiguedad"] = pd.to_numeric(bienes_parseados["Antiguedad"], errors="coerce")
bienes_parseados["Antiguedad_al_2024"] = 2024 - bienes_parseados["Antiguedad"]
bienes_parseados["superficie_inmueble_division_vertical"] = pd.to_numeric(
    bienes_parseados["superficie_inmueble_division_vertical"], errors="coerce"
)

bins = [-np.inf] + list(range(5, 105, 5)) + [np.inf]
labels = [f"{i-5}-{i} años" for i in range(5, 105, 5)]
labels.append("Mas de 100 años")

bienes_parseados["Antiguedad_categoria"] = pd.cut(
    bienes_parseados["Antiguedad_al_2024"],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# Filtro original (los que quieres EXCLUIR)
filtro_excluir = (
    (bienes_parseados["Antiguedad"] < 1800) | 
    (bienes_parseados["Antiguedad"] > 2023) |
    (bienes_parseados["superficie_inmueble_division_vertical"] == 0) |
    (bienes_parseados["superficie_inmueble_division_vertical"] > 1000)
)
# Nos quedamos con todo lo que NO cumple ese filtro
bienes_validos = bienes_parseados[~filtro_excluir]

columnas_eliminar = [
    "superficie_finca_m2",
    "id_parcela",
    "superficie_sobre_rasante",
    "superficie_bajo_sasante",
    "superficie_cubierta",
    "superficie_construida_total"
]

bienes_validos = bienes_validos.drop(columns=columnas_eliminar, errors="ignore")

bienes_validos

,id_bien,superficie_inmueble_division_vertical,Antiguedad,CUSEC,Seccion_id,anio_ultima_reforma,Antiguedad_al_2024,Antiguedad_categoria
0,0001,149,2004,0500101001,<NA>,<NA>,20,15-20 años
1,0001,220,1902,0500101001,<NA>,<NA>,122,Mas de 100 años
2,0001,219,1920,0500101001,<NA>,<NA>,104,Mas de 100 años
3,0001,206,1995,0500101001,<NA>,<NA>,29,25-30 años
4,0001,494,1910,0500101001,<NA>,<NA>,114,Mas de 100 años
...,...,...,...,...,...,...,...,...
1766747,0001,147,1966,4927503004,3457,<NA>,58,55-60 años
1766748,0001,97,1950,4927503004,3457,<NA>,74,70-75 años
1766749,0001,327,1980,4927505004,<NA>,<NA>,44,40-45 años
1766750,0001,370,1990,4927505004,<NA>,<NA>,34,30-35 años


In [25]:
# Asumimos que tu DF original se llama df_bienes
df = bienes_validos.copy()

# --- Limpieza minima ---
# Asegurar que columnas numericas estan correctamente tipadas
df["superficie_inmueble_division_vertical"] = pd.to_numeric(df["superficie_inmueble_division_vertical"], errors="coerce")
df["Antiguedad"] = pd.to_numeric(df["Antiguedad"], errors="coerce")
df["Antiguedad_limpia"] = df["Antiguedad"].replace(1900, np.nan) # Porque 1900 no es medida real
df["anio_ultima_reforma"] = pd.to_numeric(df["anio_ultima_reforma"], errors="coerce")

# Viviendas construidas antes del 2000 (susceptibles de reforma)
df["vivienda_pre2000"] = np.where(df["Antiguedad"] < 2000, 1, 0)

# Viviendas antiguas reformadas después de 2000
df["reformado_2000_valido"] = np.where(
    (df["vivienda_pre2000"].fillna(0) == 1) & 
    (df["anio_ultima_reforma"].fillna(0) >= 2000),
    1,
    0
)

# --- Función para coeficiente de variacion (desigualdad interna) ---
def coef_variacion(x):
    if x.mean() == 0:
        return 0
    return x.std(ddof=1) / x.mean()

# --- Agregacion corporativa por CUSEC ---
DH = (
    df.groupby("CUSEC")
      .agg(
          numero_bienes=("id_bien", "count"),
          superficie_media=("superficie_inmueble_division_vertical", "mean"),
          antiguedad_media=("Antiguedad_limpia", "mean"),
          dispersion_antiguedad = ("Antiguedad", coef_variacion),
          dispersion_superficie=("superficie_inmueble_division_vertical", coef_variacion),
          
          # NUEVO: parque antiguo
          viviendas_pre2000=("vivienda_pre2000", "sum"),
          viviendas_reformadas_2000=("reformado_2000_valido", "sum"),
      )
      .reset_index()
)

# Convertir % reformados a porcentaje real
DH["pct_reformado_2000"] = np.where(
    DH["viviendas_pre2000"] > 0,
    (DH["viviendas_reformadas_2000"] / DH["viviendas_pre2000"]) * 100,
    np.nan
)

print(DH.info())
DH.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3535 entries, 0 to 3534
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CUSEC                      3535 non-null   object 
 1   numero_bienes              3535 non-null   int64  
 2   superficie_media           3535 non-null   float64
 3   antiguedad_media           3534 non-null   float64
 4   dispersion_antiguedad      3535 non-null   float64
 5   dispersion_superficie      3535 non-null   float64
 6   viviendas_pre2000          3535 non-null   int64  
 7   viviendas_reformadas_2000  3535 non-null   int64  
 8   pct_reformado_2000         3510 non-null   float64
dtypes: float64(5), int64(3), object(1)
memory usage: 248.7+ KB
None


,CUSEC,numero_bienes,superficie_media,antiguedad_media,dispersion_antiguedad,dispersion_superficie,viviendas_pre2000,viviendas_reformadas_2000,pct_reformado_2000
1742,3710702007,798,167.390977,1985.383459,0.008596,0.610436,641,13,2.028081
2122,3735501001,294,212.251701,1962.932367,0.026055,0.481904,244,58,23.770492
2036,3727901001,691,184.480463,1971.451895,0.012483,0.663220,592,151,25.506757
885,0948201001,56,296.303571,1959.414634,0.018962,0.600704,51,18,35.294118
370,0904801001,1117,145.755595,1978.269196,0.010409,0.777239,1005,37,3.681592
1380,3403301001,9,494.222222,NaN,0.000000,0.417260,9,5,55.555556
2591,4217302002,900,129.257778,1976.500000,0.008689,0.359184,851,20,2.350176
2647,4700201001,16,299.687500,1938.600000,0.025986,0.683044,13,2,15.384615
2609,4217601001,32,273.125000,1987.400000,0.023516,0.385528,26,7,26.923077
1612,3423801001,311,202.520900,1956.668966,0.016865,0.647139,278,25,8.992806


In [26]:
# Formateo corporativo de KPIs

# 1. Superficie media → 3 decimales
DH["superficie_media"] = DH["superficie_media"].round(3)

# 2. Antiguedad media → entero
DH["antiguedad_media"] = DH["antiguedad_media"].round().astype("Int64")

# 3. % bienes reformados → 2 decimales
DH["pct_reformado_2000"] = DH["pct_reformado_2000"].round(2)

# 4. Disparidad de superficies → 2 decimales
DH["dispersion_superficie"] = DH["dispersion_superficie"].round(2)

# 4. Disparidad de antiguedades → 2 decimales
DH["dispersion_antiguedad"] = DH["dispersion_antiguedad"].round(2)

print(DH.info())
DH.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3535 entries, 0 to 3534
Data columns (total 9 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CUSEC                      3535 non-null   object 
 1   numero_bienes              3535 non-null   int64  
 2   superficie_media           3535 non-null   float64
 3   antiguedad_media           3534 non-null   Int64  
 4   dispersion_antiguedad      3535 non-null   float64
 5   dispersion_superficie      3535 non-null   float64
 6   viviendas_pre2000          3535 non-null   int64  
 7   viviendas_reformadas_2000  3535 non-null   int64  
 8   pct_reformado_2000         3510 non-null   float64
dtypes: Int64(1), float64(4), int64(3), object(1)
memory usage: 252.1+ KB
None


,CUSEC,numero_bienes,superficie_media,antiguedad_media,dispersion_antiguedad,dispersion_superficie,viviendas_pre2000,viviendas_reformadas_2000,pct_reformado_2000
865,0944701001,59,137.847,1959,0.02,0.38,52,20,38.46
3245,4902102004,353,110.878,1981,0.01,0.56,348,2,0.57
1627,3700501001,190,188.368,1958,0.02,0.63,176,45,25.57
2153,3738201001,171,229.357,1947,0.02,0.64,163,51,31.29
1790,3715501001,75,191.400,1959,0.03,0.49,50,25,50.00
2275,4012001001,201,245.542,1996,0.02,0.49,97,17,17.53
1414,3406901002,920,151.510,1983,0.02,0.70,688,14,2.03
2520,4209501001,1614,189.985,1999,0.02,0.56,552,57,10.33
728,0927001001,165,180.139,1951,0.02,0.52,150,26,17.33
3524,4927504003,685,144.851,1984,0.02,0.87,584,38,6.51


In [27]:
# Cargo el shp para obtener el id:
ruta_shp_secciones = os.path.join(DATA_OUTPUTS_DIR, "Shapefiles", "cyl_2022.shp")
gdf_secciones = gpd.read_file(ruta_shp_secciones)
gdf_secciones["CUSEC"] = (
    gdf_secciones["CUSEC"]
    .astype(str)
    .str.strip()
    .str.zfill(10)  # rellena con ceros a la izquierda hasta 10 caracteres
)

DH = (
    DH.merge(
        gdf_secciones.drop(columns=["geometry","NMUN","NPRO"]),
        on="CUSEC",
        how="left"
    )
)

bienes_validos = (
    bienes_validos.merge(
        gdf_secciones.drop(columns=["geometry","NMUN","NPRO"]),
        on="CUSEC",
        how="left"
    )
)
print(DH.info())
DH.sample(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3535 entries, 0 to 3534
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   CUSEC                      3535 non-null   object 
 1   numero_bienes              3535 non-null   int64  
 2   superficie_media           3535 non-null   float64
 3   antiguedad_media           3534 non-null   Int64  
 4   dispersion_antiguedad      3535 non-null   float64
 5   dispersion_superficie      3535 non-null   float64
 6   viviendas_pre2000          3535 non-null   int64  
 7   viviendas_reformadas_2000  3535 non-null   int64  
 8   pct_reformado_2000         3510 non-null   float64
 9   Seccion_id                 3535 non-null   int64  
dtypes: Int64(1), float64(4), int64(4), object(1)
memory usage: 279.8+ KB
None


,CUSEC,numero_bienes,superficie_media,antiguedad_media,dispersion_antiguedad,dispersion_superficie,viviendas_pre2000,viviendas_reformadas_2000,pct_reformado_2000,Seccion_id
298,0526001001,307,194.440,1988,0.02,0.55,242,40,16.53,296
3258,4903001001,141,351.830,1968,0.02,0.50,132,28,21.21,3194
2912,4718601006,547,143.399,1973,0.00,0.40,547,6,1.10,2853
2843,4713201001,32,247.781,1924,0.03,0.58,25,5,20.00,2784
30,0501904002,929,119.102,1982,0.01,0.67,795,17,2.14,29
2546,4213001001,30,326.933,1948,0.01,0.42,28,21,75.00,2487
146,0510601001,62,145.274,1988,0.02,0.42,53,5,9.43,144
2325,4017401001,433,204.732,1975,0.01,0.63,371,68,18.33,2268
1741,3710702006,903,144.035,1975,0.01,0.67,784,69,8.80,1686
2981,4718606003,809,101.595,1969,0.02,0.70,714,13,1.82,2922


In [28]:
# Creamos la carpeta si no existe
os.makedirs(DATA_OUTPUTS_DH, exist_ok=True)

# Rutas de salida
ruta_resultado = os.path.join(DATA_OUTPUTS_DH, "DH_Seccion.csv")
ruta_detalle = os.path.join(DATA_OUTPUTS_DH, "Viviendas.csv")

# Guardar DataFrames
DH.to_csv(
    ruta_resultado,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)
bienes_validos.to_csv(
    ruta_detalle,
    index=False,
    encoding="utf-8-sig",
    sep=";",          # separador de columnas compatible con Excel español
    decimal=",",      # separador decimal europeo
    float_format="%.3f"
)

print(f"✅ Archivos guardados correctamente en:\n- {DATA_OUTPUTS_DH}")

✅ Archivos guardados correctamente en:
- D:\MASTER EN CIENCIA DE DATOS\TFM\TrabajoFinal\ivst-tfm\data\outputs\DH_Dim_habitacional
